In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS lakehouse")
spark.sql("USE CATALOG lakehouse")
spark.sql("CREATE SCHEMA IF NOT EXISTS development")
spark.sql("USE SCHEMA development")
spark.sql("CREATE VOLUME IF NOT EXISTS bronze")



In [0]:
df = spark.read.table("samples.bakehouse.sales_transactions")
df.show()


In [0]:
path = "/Volumes/lakehouse/development/bronze/sales_transactions/"

In [0]:


df.write.format("delta").mode("overwrite").save(path)


In [0]:
display(dbutils.fs.ls(path))

In [0]:
cols_to_fix = ["transactionID", "customerID", "franchiseID"]
for col in cols_to_fix:
  spark.sql(f"ALTER TABLE delta.`{path}` ALTER COLUMN {col} SET NOT NULL")

In [0]:
checked_df = spark.read.format("delta").load(path)


In [0]:
checked_df.printSchema()


In [0]:
checked_df.describe().show()

In [0]:
checked_df.summary().show()

In [0]:
f = checked_df.select("franchiseID").distinct().count()
c = checked_df.select("customerID").distinct().count()

print(f"Distinct franchise count: {f}")
print(f"Distinct customer count:{c}")

In [0]:
checked_df = checked_df.repartition(4,"franchiseID")



In [0]:
checked_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lakehouse.development.salestransactions")